In [ ]:
import networkx as nx
import time
from scipy.io import mmread
import matplotlib.pyplot as plt




==========================
Load Graph
==========================


In [ ]:

def load_graph(file_path):

    if file_path.endswith(".mtx"):

        matrix = mmread(file_path)

        G = nx.from_scipy_sparse_array(matrix)

    elif file_path.endswith(".edges"):

        G = nx.Graph()

        with open(file_path, "r") as file:

            for line in file:

                if line.startswith("%"):
                    continue

                parts = line.split()

                if len(parts) >= 2:

                    u = int(parts[0])

                    v = int(parts[1])

                    G.add_edge(u, v)

    else:

        raise ValueError(
            f"Unsupported file format: {file_path}"
        )

    return G




==========================
PLL Algorithm
==========================


In [ ]:

class PLL:

    def __init__(self, graph, num_landmarks=10):

        self.graph = graph

        self.landmarks = list(graph.nodes())[:num_landmarks]

        self.labels = {}

    def preprocess(self):

        for landmark in self.landmarks:

            distances = nx.single_source_dijkstra_path_length(
                self.graph,
                landmark
            )

            self.labels[landmark] = distances

    def query(self, source, target):

        best_distance = float("inf")

        for landmark in self.landmarks:

            if (
                source in self.labels[landmark]
                and target in self.labels[landmark]
            ):

                distance = (
                    self.labels[landmark][source]
                    + self.labels[landmark][target]
                )

                best_distance = min(best_distance, distance)

        return best_distance




==========================
DATASETS
==========================


In [ ]:

datasets = [

    "inf-USAir97.mtx",

    "inf-power.mtx",

    "road-roadNet-CA.mtx",

    "road-euroroad.edges"
]




==========================
MAIN
==========================


In [ ]:

if __name__ == "__main__":

    print("\n===== PLL RESULTS =====\n")

    dataset_names = []

    preprocessing_times = []

    node_counts = []

    edge_counts = []

    query_times = []

    for dataset in datasets:

        print("=" * 60)

        print("Dataset:", dataset)

        G = load_graph(dataset)

        print("Nodes:", G.number_of_nodes())

        print("Edges:", G.number_of_edges())

        pll = PLL(G, num_landmarks=10)



==========================
Preprocessing Time
==========================


In [ ]:
        start = time.perf_counter()

        pll.preprocess()

        end = time.perf_counter()

        preprocess_time = end - start

        print(
            "Preprocessing Time:",
            round(preprocess_time, 6),
            "seconds"
        )



==========================
Query Test Time
==========================


In [ ]:
        nodes_list = list(G.nodes())

        if len(nodes_list) > 1:

            source = nodes_list[0]
            target = nodes_list[-1]

            q_start = time.perf_counter()

            result = pll.query(source, target)

            q_end = time.perf_counter()

            q_time = q_end - q_start

            print(
                f"Approx Distance {source} -> {target}:",
                result
            )

            print(
                "Query Time:",
                round(q_time, 6),
                "seconds"
            )

        else:
            q_time = 0

        print()



==========================
Store Results
==========================


In [ ]:
        dataset_names.append(dataset)

        preprocessing_times.append(preprocess_time)

        query_times.append(q_time)

        node_counts.append(G.number_of_nodes())

        edge_counts.append(G.number_of_edges())




==========================
Preprocessing Time Graph
==========================


In [ ]:

    plt.figure(figsize=(8, 5))

    plt.plot(
        dataset_names,
        preprocessing_times,
        marker="o"
    )

    plt.title("PLL Preprocessing Time")

    plt.xlabel("Datasets")

    plt.ylabel("Time (seconds)")

    plt.grid()

    plt.savefig("pll_preprocessing_time.png")

    plt.show()




==========================
Query Time Graph
==========================


In [ ]:

    plt.figure(figsize=(8, 5))

    plt.plot(
        dataset_names,
        query_times,
        marker="o"
    )

    plt.title("PLL Query Time")

    plt.xlabel("Datasets")

    plt.ylabel("Time (seconds)")

    plt.grid()

    plt.savefig("pll_query_time.png")

    plt.show()




==========================
Node Growth Graph
==========================


In [ ]:

    plt.figure(figsize=(8, 5))

    plt.bar(
        dataset_names,
        node_counts
    )

    plt.title("Node Growth")

    plt.xlabel("Datasets")

    plt.ylabel("Number of Nodes")

    plt.grid()

    plt.savefig("pll_node_growth.png")

    plt.show()




==========================
Edge Growth Graph
==========================


In [ ]:

    plt.figure(figsize=(8, 5))

    plt.bar(
        dataset_names,
        edge_counts
    )

    plt.title("Edge Growth")

    plt.xlabel("Datasets")

    plt.ylabel("Number of Edges")

    plt.grid()

    plt.savefig("pll_edge_growth.png")

    plt.show()
